# 1. Creating Table

## 1.1 Adding constraints

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cpt_utility_catalog.gold.fct_cpi(
    cpi_key BIGINT NOT NULL,
    date_key INT NOT NULL,
    product_key BIGINT NOT NULL,
    geography_key BIGINT NOT NULL,
    index_value DECIMAL(5,1),
    decile INT,

    CONSTRAINT pk_fct_cpi PRIMARY KEY(cpi_key) RELY,
    CONSTRAINT fk_fct_cpi_date FOREIGN KEY(date_key) REFERENCES cpt_utility_catalog.gold.dim_date(date_key) RELY,
    CONSTRAINT fk_fct_cpi_product FOREIGN KEY(product_key) REFERENCES cpt_utility_catalog.gold.dim_product(product_key) RELY,
    CONSTRAINT fk_fct_cpi_geography FOREIGN KEY(geography_key) REFERENCES cpt_utility_catalog.gold.dim_geonational(geography_key) RELY
)


## 1.2 Populating Table

In [0]:
%sql
INSERT OVERWRITE TABLE cpt_utility_catalog.gold.fct_cpi
SELECT
xxhash64(c.id) AS cpi_key,

COALESCE(CAST(date_format(c.date,'yyyyMMdd') AS INT), -1) AS date_key,
COALESCE(dp.product_key, xxhash64('unmapped')) AS product_key,
COALESCE(dg.geography_key, xxhash64('unmapped')) AS geography_key,

c.index_value AS index_value,
decile AS decile

FROM cpt_utility_catalog.silver.silver_cpi_cleaned c

LEFT JOIN cpt_utility_catalog.gold.dim_product dp
ON xxhash64(concat_ws('||', 
            LOWER(TRIM(c.series_identifier)), 
            LOWER(TRIM(c.category)), 
            LOWER(TRIM(c.subcategory)), 
            LOWER(TRIM(c.survey_code))
        )) = dp.product_key

LEFT JOIN cpt_utility_catalog.gold.dim_geonational dg
ON xxhash64(LOWER(TRIM(c.geographic_area))) = dg.geography_key

